# 01 — Google Colab: generator-model inference

Run this notebook on a **GPU Colab runtime**. It runs one 7B generator at a time and checkpoints every phase directly to the Drive run directory. Run the notebook twice, once per configured generator, preferably in separate Colab sessions.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/political-bias-lab.git"
PROFILE = "smoke"
TARGET_MODEL = "qwen2.5-7b"  # then rerun in a fresh session with "mistral-7b-v0.3"
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
REPO_DIR = "/content/political-bias-lab"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
os.chdir(REPO_DIR)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
RUN_ID = f"{PROFILE}-{GIT_SHA[:8]}"
RUN_ROOT = f"{DRIVE_ROOT}/runs/{RUN_ID}"
print("Run ID:", RUN_ID)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

In [ ]:
from src.cloud import link_colab_persistent_dirs
link_colab_persistent_dirs(REPO_DIR, RUN_ROOT)
from pathlib import Path
required = Path(REPO_DIR)/"prepared/phase1_sample.parquet"
assert required.exists(), f"Prepared data not found at {required}. Run notebook 00 on the same Git commit/profile first."

In [ ]:
from src.config import load_config
from src.pipeline import model_config_by_short_name
from src.runtime import require_colab_gpu
from pathlib import Path

cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")
model_cfg = model_config_by_short_name(cfg, TARGET_MODEL)
info = require_colab_gpu(float(model_cfg.get("min_gpu_vram_gb", 10)))
print(info)
print("Model:", model_cfg["id"])

## Run all four generator-side phases

This includes Phase 1 sentiment scoring, the Phase 2 persona capability anchor + susceptibility experiment, hierarchical few-shot Phase 3 classification, and Phase 4 baseline/adaptive response generation. The 4-bit model is unloaded at the end.

In [ ]:
from src.pipeline import run_one_model
from src.io_utils import write_json
from pathlib import Path

metadata = run_one_model(cfg, model_cfg, root=REPO_DIR)
write_json(metadata, Path(REPO_DIR)/f"results/manifests/generator_{TARGET_MODEL}.json")
metadata

In [ ]:
import pandas as pd
from pathlib import Path
raw_dir = Path(REPO_DIR)/"results/raw"
for p in sorted(raw_dir.glob("*.parquet")):
    df = pd.read_parquet(p)
    if "model" in df.columns:
        df = df[df["model"] == TARGET_MODEL]
    elif "generator_model" in df.columns:
        df = df[df["generator_model"] == TARGET_MODEL]
    failures = int(df["error"].notna().sum()) if "error" in df.columns else 0
    print(f"{p.name}: rows={len(df):,}, failures={failures:,}")

### Next

After **both** `qwen2.5-7b` and `mistral-7b-v0.3` have completed for the same Run ID, open `02_colab_phase4_judge.ipynb`. The second generator run resumes the same Drive checkpoints instead of overwriting the first model.